# 03 · CatBoost Feature View

**Competition:** Kaggle Playground Series S6E9 — Predicting Electric Vehicle Purchases  
**Metric:** ROC-AUC

This notebook develops a CatBoost feature view designed to preserve categorical structure and provide prediction diversity relative to the LightGBM models.

The model combines:

1. mixed numeric and categorical base features;
2. income digit decomposition;
3. quantized income and commute features;
4. four explicit categorical interaction keys;
5. a fold-fitted 600-quantile income bin;
6. fold-local frequency features;
7. three cross-fitted Target Encodings (`auto`, `10`, `100`);
8. one additional selected-key Target Encoding with smoothing `5`;
9. CatBoost training with explicit categorical features;
10. 5-fold OOF validation with two model seeds.

The purpose of this view is not only standalone predictive strength, but also a different treatment of categorical information that can contribute complementary ranking information to the ensemble.

## 1. Imports and configuration

The notebook expects the competition files in `DATA_DIR`:

- `train.csv`
- `test.csv`
- `sample_submission.csv`

No original external dataset is required for this CatBoost view; it uses competition data only.

For reproduction of the original competition run, GPU training is preferred. Change `TASK_TYPE` to `"CPU"` if a GPU is unavailable.

In [ ]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier

from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder

# -----------------------------
# Paths
# -----------------------------
DATA_DIR = Path(".")
OUTPUT_DIR = Path("outputs/03_catboost")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Experiment setup
# -----------------------------
TARGET = "Will_Buy_EV"
ID_COL = "id"
INCOME = "Annual_Income_USD"
COMMUTE = "Daily_Commute_km"

OUTER_FOLDS = 5
INNER_FOLDS = 5
FOLD_SEED = 42
MODEL_SEEDS = [42, 152]

TASK_TYPE = "GPU"   # change to "CPU" if needed
THREADS = 8

CATS = [
    "Gender",
    "City_Type",
    "Current_Car_Type",
    "Home_Charging_Possible",
    "Subsidy_Available",
    "Range_Anxiety_Level",
]

NUMS = [
    "Age",
    INCOME,
    COMMUTE,
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
]

RAW = NUMS + CATS

## 2. Load the competition data

The binary target is mapped from `No/Yes` to `0/1`.

The CatBoost feature view uses the full competition feature set, including `Number_of_Cars_Owned`.

In [ ]:
def map_target(series):
    mapped = (
        series.astype("string")
        .str.strip()
        .str.lower()
        .map({
            "yes": 1,
            "no": 0,
            "1": 1,
            "0": 0,
            "1.0": 1,
            "0.0": 0,
        })
    )

    if mapped.isna().any():
        raise ValueError("Target contains values outside Yes/No or 0/1.")

    return mapped.astype("int8")


def canonical_features(frame):
    out = frame[RAW].copy()

    for col in CATS:
        out[col] = (
            out[col]
            .astype("string")
            .fillna("__MISSING__")
        )

    for col in NUMS:
        out[col] = pd.to_numeric(
            out[col],
            errors="raise",
        ).astype("float64")

    return out


train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(
    DATA_DIR / "sample_submission.csv"
)

y = map_target(train[TARGET]).to_numpy()

all_comp = canonical_features(
    pd.concat(
        [train[RAW], test[RAW]],
        ignore_index=True,
    )
)

n_train = len(train)
n_total = len(all_comp)
test_idx = np.arange(n_train, n_total)

print("Train rows:", len(train))
print("Test rows:", len(test))
print("Positive rate:", y.mean())

## 3. Static feature construction

The CatBoost view deliberately differs from the LightGBM exact view.

### 3.1 Base variables

All numeric variables are retained as numeric features.

Categorical variables are integer-coded and later passed to CatBoost through `cat_features`:

- `Gender`
- `City_Type`
- `Current_Car_Type`
- `Home_Charging_Possible`
- `Subsidy_Available`
- `Range_Anxiety_Level`

The integer codes are only category labels; their numeric magnitude is not intended to carry meaning because CatBoost is explicitly told which columns are categorical.

### 3.2 Income digits

The first five integer digits of annual income are added:

$$
d_p(x)
=
\left\lfloor
\frac{x}{10^p}
\right\rfloor
\bmod 10,
\qquad
p=0,\ldots,4.
$$

### 3.3 Quantized numeric keys

The view adds:

- `income_floor_1`
- `income_floor_100`
- `income_floor_1000`
- `commute_integer`

### 3.4 Structural indicators

The selected feature specification also includes:

- `homecharge`
- `is_30k_spike`
- `is_income_cliff`
- `is_low_income_interval`
- `is_env_one`

### 3.5 Explicit categorical interactions

Four interaction keys are created:

- `Environmental_Concern_Level × Subsidy_Available`
- `Home_Charging_Possible × Range_Anxiety_Level`
- `City_Type × Home_Charging_Possible`
- `Environmental_Concern_Level × Range_Anxiety_Level`

These interaction columns are also treated as categorical by CatBoost.

In [ ]:
def remove_constant_and_duplicate_columns(frame):
    kept = []
    seen = {}

    for name in frame.columns:
        values = frame[name].to_numpy()

        if frame[name].nunique(dropna=False) <= 1:
            continue

        digest = hashlib.blake2b(
            np.ascontiguousarray(values).tobytes(),
            digest_size=12,
        ).digest()

        duplicate = any(
            np.array_equal(
                values,
                frame[old].to_numpy(),
                equal_nan=True,
            )
            for old in seen.get(digest, [])
        )

        if duplicate:
            continue

        seen.setdefault(digest, []).append(name)
        kept.append(name)

    return frame[kept]


numeric_features = {}
raw_keys = {}

def add_key(name, values):
    raw_keys[name] = np.asarray(values)


# -----------------------------
# Base numeric + categorical variables
# -----------------------------
for col in RAW:
    values = all_comp[col]

    if col in NUMS:
        numeric_features[col] = (
            values.to_numpy(dtype="float32")
        )

    else:
        if col in [
            "Home_Charging_Possible",
            "Subsidy_Available",
        ]:
            encoded = (
                values == "Yes"
            ).to_numpy(dtype="int32")

        elif col == "Range_Anxiety_Level":
            encoded = (
                values
                .map({
                    "Low": 0,
                    "Medium": 1,
                    "High": 2,
                })
                .to_numpy(dtype="int32")
            )

        else:
            encoded = pd.factorize(
                values,
                sort=True,
            )[0].astype("int32")

        numeric_features[col] = encoded

    add_key(
        col,
        values.to_numpy(),
    )


income = all_comp[INCOME].to_numpy()
commute = all_comp[COMMUTE].to_numpy()

# -----------------------------
# Income digits: powers 0..4
# -----------------------------
for power in range(5):
    name = f"income_digit_{power}"

    digit = (
        income.astype("int64")
        // (10 ** power)
        % 10
    ).astype("float32")

    numeric_features[name] = digit
    add_key(name, digit)

# -----------------------------
# Exact-style income ladder
# -----------------------------
for divisor in [1, 100, 1000]:
    name = f"income_floor_{divisor}"

    floor_value = np.floor(
        income / divisor
    ).astype("float32")

    numeric_features[name] = floor_value
    add_key(name, floor_value)

# Integer commute
commute_integer = np.floor(
    commute
).astype("float32")

numeric_features["commute_integer"] = commute_integer
add_key(
    "commute_integer",
    commute_integer,
)

# -----------------------------
# Structural indicators
# -----------------------------
env = all_comp[
    "Environmental_Concern_Level"
].to_numpy()

home = (
    all_comp["Home_Charging_Possible"] == "Yes"
).to_numpy(dtype="int8")

numeric_features["homecharge"] = (
    all_comp["Charging_Stations_Near_Home"].to_numpy()
    + 5 * home
).astype("float32")

numeric_features["is_30k_spike"] = (
    income == 30000
).astype("float32")

numeric_features["is_income_cliff"] = (
    income >= 170537
).astype("float32")

numeric_features["is_low_income_interval"] = (
    (income >= 38000)
    & (income <= 42000)
).astype("float32")

numeric_features["is_env_one"] = (
    env == 1
).astype("float32")

# -----------------------------
# Explicit categorical interaction keys
# -----------------------------
interaction_pairs = [
    (
        "Environmental_Concern_Level",
        "Subsidy_Available",
    ),
    (
        "Home_Charging_Possible",
        "Range_Anxiety_Level",
    ),
    (
        "City_Type",
        "Home_Charging_Possible",
    ),
    (
        "Environmental_Concern_Level",
        "Range_Anxiety_Level",
    ),
]

interaction_names = []

for left, right in interaction_pairs:
    name = f"{left}__{right}"

    values = (
        all_comp[left].astype(str)
        + "|"
        + all_comp[right].astype(str)
    )

    numeric_features[name] = pd.factorize(
        values,
        sort=True,
    )[0].astype("int32")

    add_key(
        name,
        values.to_numpy(),
    )

    interaction_names.append(name)


static_features = remove_constant_and_duplicate_columns(
    pd.DataFrame(numeric_features)
)

# Encode target-encoding / frequency keys with arbitrary integer labels.
key_codes = {}

for name, values in raw_keys.items():
    codes, uniques = pd.factorize(
        pd.Series(values),
        sort=False,
    )

    if len(uniques) > 1:
        key_codes[name] = codes.astype("int32")

keys = remove_constant_and_duplicate_columns(
    pd.DataFrame(key_codes)
)

cat_feature_names = [
    col
    for col in static_features.columns
    if col in CATS or col in interaction_names
]

print("Static features:", static_features.shape[1])
print("Encoding keys:", keys.shape[1])
print("Categorical model features:", cat_feature_names)

## 4. Fold-specific feature construction

The following transformations are fitted separately inside each outer fold.

### 4.1 600-quantile income bin

Quantile edges are estimated only from outer-training income values. The resulting `income_quantile_600` feature is used:

- directly as a numeric feature;
- as an additional frequency key;
- as an additional Target-Encoding key.

### 4.2 Fold-local frequency

For every encoding key $k$,

$$
\mathrm{freq}_{\mathrm{fold}}(v)
=
\frac{
\#\{i\in\mathrm{outer\ train}: k_i=v\}
}{
n_{\mathrm{outer\ train}}
}.
$$

Validation and test rows are mapped with the outer-training frequency table. Unseen values receive `0`.

### 4.3 Triple cross-fitted Target Encoding

Every key receives:

- `smooth="auto"`
- `smooth=10`
- `smooth=100`

Training rows receive inner cross-fitted encodings; outer validation and test are transformed using an encoder fitted only on outer-training labels.

### 4.4 Extra selected-key Target Encoding

A smaller selected key set receives an additional Target Encoding with `smooth=5`:

- income
- commute
- age
- `income_floor_1000`
- `commute_integer`
- `Environmental_Concern_Level__Subsidy_Available`
- `Environmental_Concern_Level__Range_Anxiety_Level`
- `City_Type__Home_Charging_Possible`

This additional encoding is part of the selected CatBoost feature view.

In [ ]:
def crossfit_target_encoding(
    key_frame,
    y,
    train_idx,
    valid_idx,
    test_idx,
    smooth_values,
    inner_folds,
    seed,
):
    train_parts = []
    valid_parts = []
    test_parts = []
    feature_names = []

    for smooth in smooth_values:
        encoder = TargetEncoder(
            target_type="binary",
            smooth=smooth,
            cv=inner_folds,
            shuffle=True,
            random_state=seed,
        )

        train_encoded = encoder.fit_transform(
            key_frame.iloc[train_idx],
            y[train_idx],
        ).astype("float32")

        valid_encoded = encoder.transform(
            key_frame.iloc[valid_idx]
        ).astype("float32")

        test_encoded = encoder.transform(
            key_frame.iloc[test_idx]
        ).astype("float32")

        train_parts.append(train_encoded)
        valid_parts.append(valid_encoded)
        test_parts.append(test_encoded)

        feature_names.extend([
            f"TE_{col}_{smooth}"
            for col in key_frame.columns
        ])

    return (
        np.column_stack(train_parts),
        np.column_stack(valid_parts),
        np.column_stack(test_parts),
        feature_names,
    )


def build_fold_matrices(
    train_idx,
    valid_idx,
    seed,
):
    test_idx_local = test_idx

    X_train = static_features.iloc[
        train_idx
    ].copy()

    X_valid = static_features.iloc[
        valid_idx
    ].copy()

    X_test = static_features.iloc[
        test_idx_local
    ].copy()

    fold_keys = keys.copy()

    # -----------------------------
    # Fold-fitted 600-quantile income bin
    # -----------------------------
    edges = np.unique(
        np.quantile(
            income[train_idx],
            np.linspace(
                0,
                1,
                601,
            ),
        )
    )

    income_quantile_600 = np.searchsorted(
        edges[1:-1],
        income,
        side="right",
    ).astype("int32")

    fold_keys[
        "income_quantile_600"
    ] = income_quantile_600

    X_train[
        "income_quantile_600"
    ] = income_quantile_600[train_idx]

    X_valid[
        "income_quantile_600"
    ] = income_quantile_600[valid_idx]

    X_test[
        "income_quantile_600"
    ] = income_quantile_600[test_idx_local]

    # -----------------------------
    # Fold-local frequency features
    # -----------------------------
    for col in fold_keys.columns:
        freq_map = (
            fold_keys.iloc[train_idx][col]
            .value_counts(normalize=True)
        )

        X_train[f"freq_{col}"] = (
            fold_keys.iloc[train_idx][col]
            .map(freq_map)
            .fillna(0)
            .to_numpy(dtype="float32")
        )

        X_valid[f"freq_{col}"] = (
            fold_keys.iloc[valid_idx][col]
            .map(freq_map)
            .fillna(0)
            .to_numpy(dtype="float32")
        )

        X_test[f"freq_{col}"] = (
            fold_keys.iloc[test_idx_local][col]
            .map(freq_map)
            .fillna(0)
            .to_numpy(dtype="float32")
        )

    # -----------------------------
    # Triple TE: auto / 10 / 100
    # -----------------------------
    (
        te_train,
        te_valid,
        te_test,
        te_names,
    ) = crossfit_target_encoding(
        key_frame=fold_keys,
        y=y,
        train_idx=train_idx,
        valid_idx=valid_idx,
        test_idx=test_idx_local,
        smooth_values=[
            "auto",
            10.0,
            100.0,
        ],
        inner_folds=INNER_FOLDS,
        seed=seed,
    )

    X_train[te_names] = te_train
    X_valid[te_names] = te_valid
    X_test[te_names] = te_test

    # -----------------------------
    # Extra selected-key TE, smooth=5
    # -----------------------------
    selected_keys = [
        col
        for col in [
            INCOME,
            COMMUTE,
            "Age",
            "income_floor_1000",
            "commute_integer",
            "Environmental_Concern_Level__Subsidy_Available",
            "Environmental_Concern_Level__Range_Anxiety_Level",
            "City_Type__Home_Charging_Possible",
        ]
        if col in fold_keys.columns
    ]

    (
        extra_train,
        extra_valid,
        extra_test,
        extra_names,
    ) = crossfit_target_encoding(
        key_frame=fold_keys[
            selected_keys
        ],
        y=y,
        train_idx=train_idx,
        valid_idx=valid_idx,
        test_idx=test_idx_local,
        smooth_values=[5.0],
        inner_folds=INNER_FOLDS,
        seed=seed,
    )

    X_train[extra_names] = extra_train
    X_valid[extra_names] = extra_valid
    X_test[extra_names] = extra_test

    # Remove constants according to outer-training rows only.
    keep = [
        col
        for col in X_train.columns
        if X_train[col].nunique(
            dropna=False
        ) > 1
    ]

    X_train = X_train[keep]
    X_valid = X_valid[keep]
    X_test = X_test[keep]

    fold_cat_features = [
        col
        for col in cat_feature_names
        if col in keep
    ]

    for frame in [
        X_train,
        X_valid,
        X_test,
    ]:
        for col in fold_cat_features:
            frame[col] = frame[
                col
            ].astype("int32")

    return (
        X_train,
        X_valid,
        X_test,
        fold_cat_features,
    )

## 5. CatBoost model

The selected CatBoost configuration is:

- iterations: `6000`
- depth: `7`
- learning rate: `0.021673771484666256`
- L2 leaf regularization: `13.21397574842675`
- random strength: `3.4165471601740314`
- border count: `250`
- one-hot max size: `2`
- Bernoulli bootstrap
- subsample: `0.9557182534037572`
- early stopping: `300`

The objective is binary log loss, while AUC is used as the validation metric.

The explicit categorical columns are passed through `cat_features`.

In [ ]:
def fit_catboost(
    X_train,
    y_train,
    X_valid,
    y_valid,
    X_test,
    cat_features,
    seed,
):
    model = CatBoostClassifier(
        iterations=6000,
        depth=7,
        learning_rate=0.021673771484666256,
        l2_leaf_reg=13.21397574842675,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=seed,
        task_type=TASK_TYPE,
        thread_count=THREADS,
        allow_writing_files=False,
        random_strength=3.4165471601740314,
        border_count=250,
        one_hot_max_size=2,
        bootstrap_type="Bernoulli",
        subsample=0.9557182534037572,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(
            X_valid,
            y_valid,
        ),
        early_stopping_rounds=300,
        use_best_model=True,
        verbose=500,
    )

    valid_pred = model.predict_proba(
        X_valid
    )[:, 1]

    test_pred = model.predict_proba(
        X_test
    )[:, 1]

    importance = pd.DataFrame({
        "feature": X_train.columns,
        "importance": model.feature_importances_,
    })

    return (
        valid_pred,
        test_pred,
        model.tree_count_,
        importance,
    )

## 6. Five-fold OOF training with two seeds

The outer folds are fixed once:

$$
\mathrm{StratifiedKFold}
(
5,
\mathrm{shuffle=True},
\mathrm{random\_state}=42
).
$$

The original CatBoost component uses model seeds `42` and `152`.

For each seed/fold pair, all supervised feature transforms are rebuilt inside the fold before CatBoost is trained.

In [ ]:
outer_cv = StratifiedKFold(
    n_splits=OUTER_FOLDS,
    shuffle=True,
    random_state=FOLD_SEED,
)

folds = list(
    outer_cv.split(
        train,
        y,
    )
)

fold_id = np.empty(
    len(train),
    dtype="int8",
)

for fold, (_, valid_idx) in enumerate(
    folds,
    start=1,
):
    fold_id[valid_idx] = fold


seed_oof_predictions = []
seed_test_fold_predictions = []
training_rows = []
importance_frames = []

for seed in MODEL_SEEDS:
    seed_oof = np.zeros(
        len(train),
        dtype="float64",
    )

    seed_test_folds = []

    for fold, (
        train_idx,
        valid_idx,
    ) in enumerate(
        folds,
        start=1,
    ):
        (
            X_train,
            X_valid,
            X_test,
            cat_features,
        ) = build_fold_matrices(
            train_idx=train_idx,
            valid_idx=valid_idx,
            seed=seed,
        )

        (
            valid_pred,
            test_pred,
            best_iteration,
            importance,
        ) = fit_catboost(
            X_train=X_train,
            y_train=y[train_idx],
            X_valid=X_valid,
            y_valid=y[valid_idx],
            X_test=X_test,
            cat_features=cat_features,
            seed=seed,
        )

        seed_oof[
            valid_idx
        ] = valid_pred

        seed_test_folds.append(
            test_pred
        )

        fold_auc = roc_auc_score(
            y[valid_idx],
            valid_pred,
        )

        training_rows.append({
            "seed": seed,
            "fold": fold,
            "auc": fold_auc,
            "best_iteration": best_iteration,
            "n_features": X_train.shape[1],
        })

        importance["seed"] = seed
        importance["fold"] = fold
        importance_frames.append(
            importance
        )

        print(
            f"seed={seed} "
            f"fold={fold} "
            f"AUC={fold_auc:.6f} "
            f"best={best_iteration} "
            f"features={X_train.shape[1]}"
        )

    seed_oof_predictions.append(
        seed_oof
    )

    seed_test_fold_predictions.append(
        np.stack(seed_test_folds)
    )

## 7. Aggregate seeds and build rank predictions

Raw OOF predictions are averaged across the two seeds.

For later rank ensembles, OOF predictions are normalized **inside each outer fold**:

$$
R_i
=
\frac{
\operatorname{rank}(p_i)-0.5
}{
n_{\mathrm{fold}}
}.
$$

Test predictions are first averaged across seeds while retaining one prediction vector per outer fold. Each fold-level test vector is rank-normalized, then the five ranked vectors are averaged.

In [ ]:
def normalized_rank(values):
    values = np.asarray(values)

    return (
        rankdata(
            values,
            method="average",
        )
        - 0.5
    ) / len(values)


# Average raw OOF predictions across seeds.
oof_pred = np.mean(
    seed_oof_predictions,
    axis=0,
)

# Shape: seeds x folds x test_rows
test_seed_fold = np.stack(
    seed_test_fold_predictions
)

# Average seeds first, retaining fold-specific test predictions.
test_fold_pred = np.mean(
    test_seed_fold,
    axis=0,
)

# Raw test prediction.
test_pred = np.mean(
    test_fold_pred,
    axis=0,
)

# Fold-normalized OOF ranks.
oof_rank = np.zeros(
    len(train),
    dtype="float64",
)

test_rank = np.zeros(
    len(test),
    dtype="float64",
)

for fold, (_, valid_idx) in enumerate(
    folds
):
    oof_rank[
        valid_idx
    ] = normalized_rank(
        oof_pred[valid_idx]
    )

    test_rank += (
        normalized_rank(
            test_fold_pred[fold]
        )
        / OUTER_FOLDS
    )


raw_oof_auc = roc_auc_score(
    y,
    oof_pred,
)

rank_oof_auc = roc_auc_score(
    y,
    oof_rank,
)

fold_metrics = pd.DataFrame(
    training_rows
)

print(f"Raw OOF AUC:  {raw_oof_auc:.6f}")
print(f"Rank OOF AUC: {rank_oof_auc:.6f}")
print()
print("Reproduction target:")
print("  raw OOF  ≈ 0.946126")
print("  rank OOF ≈ 0.946131")

## 8. Seed stability

The two seed models were close in OOF performance:

- seed 42: reference OOF approximately `0.946084`
- seed 152: reference OOF approximately `0.946080`

Their average improved to approximately `0.946126`, which is consistent with a modest variance-reduction benefit.

In [ ]:
seed_summary = []

for seed, seed_oof in zip(
    MODEL_SEEDS,
    seed_oof_predictions,
):
    fold_scores = []

    for _, valid_idx in folds:
        fold_scores.append(
            roc_auc_score(
                y[valid_idx],
                seed_oof[valid_idx],
            )
        )

    seed_summary.append({
        "seed": seed,
        "mean_fold_auc": np.mean(
            fold_scores
        ),
        "fold_auc_std": np.std(
            fold_scores
        ),
        "pooled_oof_auc": roc_auc_score(
            y,
            seed_oof,
        ),
    })

seed_summary = pd.DataFrame(
    seed_summary
)

seed_summary

## 9. Feature importance

CatBoost feature importance is averaged across all seed/fold models.

This is descriptive only. No feature is selected or removed based on validation importance in this notebook.

In [ ]:
importance_all = pd.concat(
    importance_frames,
    ignore_index=True,
)

importance_summary = (
    importance_all
    .groupby(
        "feature",
        as_index=False,
    )["importance"]
    .mean()
    .sort_values(
        "importance",
        ascending=False,
    )
)

importance_summary.head(30)

## 10. Save reproducible outputs

These files are designed for later ensemble analysis.

In [ ]:
oof_output = pd.DataFrame({
    ID_COL: train[ID_COL],
    "y_true": y,
    "fold": fold_id,
    "prediction": oof_pred,
    "prediction_rank": oof_rank,
})

test_output = pd.DataFrame({
    ID_COL: test[ID_COL],
    "prediction": test_pred,
    "prediction_rank": test_rank,
})

submission = sample_submission.copy()
submission[TARGET] = test_pred

oof_output.to_csv(
    OUTPUT_DIR / "oof_catboost.csv",
    index=False,
)

test_output.to_csv(
    OUTPUT_DIR / "test_catboost.csv",
    index=False,
)

fold_metrics.to_csv(
    OUTPUT_DIR / "fold_metrics.csv",
    index=False,
)

seed_summary.to_csv(
    OUTPUT_DIR / "seed_metrics.csv",
    index=False,
)

importance_summary.to_csv(
    OUTPUT_DIR / "feature_importance.csv",
    index=False,
)

submission.to_csv(
    OUTPUT_DIR / "submission_catboost.csv",
    index=False,
)

print(
    "Saved outputs to:",
    OUTPUT_DIR,
)

## 11. Interpretation

The CatBoost component was not simply “the same features with another tree library”.

Its view differed from the LightGBM exact component in several ways:

- all original categorical variables were explicitly passed as categorical features;
- four categorical interaction keys were added;
- frequency statistics were fold-local rather than global train+test frequencies;
- the income digit representation was narrower (`0..4`);
- an additional selected-key Target Encoding with smoothing `5` was included;
- CatBoost used ordered categorical processing internally in addition to the explicit external encodings.

The component's value later came from both standalone strength and ensemble complementarity.

The methodological constraint remains the same as in the rest of the project: all label-derived competition features are built inside the validation structure.

## Result

- Raw OOF AUC: **0.946122**
- Rank-normalized OOF AUC: **0.946125**

This component is used in the final ensemble analysis in
`06_final_research.ipynb`.